# 📓 Agentic Evaluation Cookbook

Build a small ReAct-style agent that answers questions using two tools (a
calculator and a fact lookup), instrument it with TruLens, and evaluate its
behavior with all 7 agentic evaluators:

1. **Plan Quality** — is the agent's plan sound?
2. **Plan Adherence** — did the agent follow its plan?
3. **Execution Efficiency** — did the agent avoid redundant steps?
4. **Logical Consistency** — are the agent's steps and conclusions consistent?
5. **Tool Selection** — did the agent pick the right tool for each step?
6. **Tool Calling** — were the tools called with correct arguments?
7. **Tool Quality** — did the tool results support the final answer?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/truera/trulens/blob/main/examples/cookbooks/agentic_evaluations.ipynb)

In [ ]:
!pip install trulens trulens-providers-openai openai -q

In [ ]:
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = "sk-proj-..."

## 1. Define tools

The agent has access to a calculator (arithmetic only, safely evaluated via
`ast` rather than `eval`) and a small fact lookup table.

In [ ]:
import ast
import operator

_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}


def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](
            _eval_node(node.left), _eval_node(node.right)
        )
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")


def calculator(expression: str) -> str:
    """Safely evaluate a basic arithmetic expression."""
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_eval_node(tree.body))
    except Exception as e:
        return f"Error: {e}"


KNOWLEDGE_BASE = {
    "boiling point of water": "100 degrees Celsius at sea level",
    "speed of light": "299,792,458 meters per second",
    "population of france": "about 68 million (2024 estimate)",
}


def lookup_fact(topic: str) -> str:
    """Look up a known fact by topic."""
    return KNOWLEDGE_BASE.get(
        topic.lower().strip(), f"No fact found for '{topic}'."
    )


TOOL_IMPLS = {"calculator": calculator, "lookup_fact": lookup_fact}

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a basic arithmetic expression, e.g. '12 * (4 + 1)'.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_fact",
            "description": "Look up a known fact by topic, e.g. 'speed of light'.",
            "parameters": {
                "type": "object",
                "properties": {"topic": {"type": "string"}},
                "required": ["topic"],
            },
        },
    },
]

## 2. Build the ReAct agent

Each planning call and tool call is instrumented with `@instrument` so the
full reasoning trace is captured, not just the final answer.

In [ ]:
import json

from openai import OpenAI as OpenAIClient
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

client = OpenAIClient()

MAX_STEPS = 5


class ReActAgent:
    SYSTEM_PROMPT = (
        "You are a ReAct-style agent. Use the available tools to answer the "
        "user's question step by step. Call a tool whenever you need a fact "
        "or a calculation; otherwise respond with the final answer directly."
    )

    @instrument(span_type=SpanAttributes.SpanType.TOOL)
    def call_tool(self, name: str, arguments: dict) -> str:
        return TOOL_IMPLS[name](**arguments)

    @instrument(span_type=SpanAttributes.SpanType.GENERATION)
    def plan_step(self, messages: list):
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        return response.choices[0].message

    @instrument(
        span_type=SpanAttributes.SpanType.RECORD_ROOT,
        attributes={
            SpanAttributes.RECORD_ROOT.INPUT: "query",
            SpanAttributes.RECORD_ROOT.OUTPUT: "return",
        },
    )
    def run(self, query: str) -> str:
        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": query},
        ]
        for _ in range(MAX_STEPS):
            message = self.plan_step(messages)
            if not message.tool_calls:
                return message.content or ""
            messages.append(message.model_dump(exclude_none=True))
            for tool_call in message.tool_calls:
                arguments = json.loads(tool_call.function.arguments)
                result = self.call_tool(tool_call.function.name, arguments)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })
        return "Could not reach a final answer within the step limit."


agent = ReActAgent()

## 3. Set up TruLens logging

In [ ]:
from trulens.core import TruSession

session = TruSession()
session.reset_database()

## 4. Add the 7 agentic evaluators

Each evaluator is a `Metric` bound to the full trace via
`Selector(trace_level=True)`, so the LLM judge sees every planning and tool
call step, not just the final input/output.

In [ ]:
from trulens.core import Metric
from trulens.core import Selector
from trulens.providers.openai import OpenAI

provider = OpenAI(model_engine="gpt-4.1")
trace_selector = {"trace": Selector(trace_level=True)}

f_plan_quality = Metric(
    implementation=provider.plan_quality_with_cot_reasons,
    name="Plan Quality",
).on(trace_selector)

f_plan_adherence = Metric(
    implementation=provider.plan_adherence_with_cot_reasons,
    name="Plan Adherence",
).on(trace_selector)

f_execution_efficiency = Metric(
    implementation=provider.execution_efficiency_with_cot_reasons,
    name="Execution Efficiency",
).on(trace_selector)

f_logical_consistency = Metric(
    implementation=provider.logical_consistency_with_cot_reasons,
    name="Logical Consistency",
).on(trace_selector)

f_tool_selection = Metric(
    implementation=provider.tool_selection_with_cot_reasons,
    name="Tool Selection",
).on(trace_selector)

f_tool_calling = Metric(
    implementation=provider.tool_calling_with_cot_reasons,
    name="Tool Calling",
).on(trace_selector)

f_tool_quality = Metric(
    implementation=provider.tool_quality_with_cot_reasons,
    name="Tool Quality",
).on(trace_selector)

feedbacks = [
    f_plan_quality,
    f_plan_adherence,
    f_execution_efficiency,
    f_logical_consistency,
    f_tool_selection,
    f_tool_calling,
    f_tool_quality,
]

## 5. Register the agent

In [ ]:
from trulens.apps.app import TruApp

tru_agent = TruApp(
    agent,
    app_name="ReAct Agentic Eval Demo",
    app_version="base",
    feedbacks=feedbacks,
)

## 6. Run the agent

In [ ]:
test_queries = [
    "What is the boiling point of water, in Fahrenheit?",
    "What is the speed of light divided by 1000?",
    "What is the population of France plus 12345?",
]

with tru_agent as recording:
    for query in test_queries:
        answer = agent.run(query)
        print(f"Q: {query}")
        print(f"A: {answer}\n")

## 7. View evaluation results

You may need to run this cell a few times to see full results, as the LLM
judge evaluations take time to compute.

In [ ]:
session.get_leaderboard()

## 8. Visualize traces in the dashboard

The dashboard renders each recorded trace — planning steps, tool calls, and
the final answer — alongside the score each agentic evaluator assigned.

In [ ]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

## Summary

The 7 agentic evaluators all read the full recorded trace rather than a
single input/output pair, which lets them judge things a record-level
feedback function can't: whether the agent's plan made sense, whether it
stuck to that plan, whether it picked the right tool at each step, and
whether the tool outputs actually supported the final answer.